# PyTorchのテンソル（tensor）の扱い方①

このノートブックでは、PyTorch のテンソルを扱うときに **つまずきやすいポイント** を、小さい例と一緒にまとめています。強化学習などのコードを読むときにもよく出てくるので、ここを押さえておくと読みやすくなります。

## このノートブックで学ぶこと（目次）

1. **`torch.arange`** … 連番のテンソルを作る関数。名前の由来（array range）も簡単に紹介します。
2. **ストライド（stride）** … テンソルがメモリ上で「どの歩幅で読むか」を表す考え方です。
3. **メモリ上で「連続（contiguous）」とは** … 転置やスライス・`permute` などで non-contiguous になるパターンと、`view` が使えない理由のイメージを押さえます。
4. **`view` と `reshape`** … 形を変える 2 つの方法と、その違い（連続でないときにどうなるか）を説明します。
5. **`gather`** … インデックスに応じてテンソルから値を取り出すメソッド。`torch.gather` と `tensor.gather` の両方の書き方と、強化学習での典型的な使い方（選んだ行動の Q 値だけを取り出す）を扱います。

## `torch.arange` 関数（連番のテンソルを作る）

`torch.arange` は、**一定間隔の連番を持つ 1 次元テンソル** を作る関数です。NumPy の `np.arange` とほぼ同じイメージです。

- 書式: `torch.arange(start, end, step)`
  - **`start`**: 開始値（含む）
  - **`end`**: 終了値（**含まない**）
  - **`step`**: 増分（省略時は 1）
- よくある省略形:
  - `torch.arange(n)` → `0, 1, 2, ..., n-1` のテンソル

### 名前の由来（arange って何？）

`arange` は、**array range（配列として返す range）** を縮めた名前だと考えると分かりやすいです。

- `range` は Python の `range()` と同じく「連続した整数の並び」を表現するイメージ
- その「range」を **array（配列）として返す → array range → arange**

NumPy にもまったく同じ名前・役割の `np.arange` があり、PyTorch の `torch.arange` もそれに対応する関数だと思っておけば OK です。

In [13]:
import torch

# 0 以上 5 未満（0,1,2,3,4）
a = torch.arange(5)
print("a = torch.arange(5):", a)

# 1 以上 6 未満（1,2,3,4,5）
b = torch.arange(1, 6)
print("\nb = torch.arange(1, 6):", b)

# 0 以上 10 未満を 2 ずつ（0,2,4,6,8）
c = torch.arange(0, 10, 2)
print("\nc = torch.arange(0, 10, 2):", c)

# 少数ステップも指定できる（ただし浮動小数の誤差には注意）
d = torch.arange(0.0, 1.0, 0.2)
print("\nd = torch.arange(0.0, 1.0, 0.2):", d)

a = torch.arange(5): tensor([0, 1, 2, 3, 4])

b = torch.arange(1, 6): tensor([1, 2, 3, 4, 5])

c = torch.arange(0, 10, 2): tensor([0, 2, 4, 6, 8])

d = torch.arange(0.0, 1.0, 0.2): tensor([0.0000, 0.2000, 0.4000, 0.6000, 0.8000])


## テンソルの「ストライド」とは？

PyTorch のテンソルは、中身の値を **1 本のメモリ配列** に並べたうえで、「どのくらい飛ばしながら読むか」というルールを **ストライド（stride）** として持っています。

- **`shape`（サイズ）**: 各次元の長さ（何行×何列×…か）
- **`stride`**: その次元の **隣の要素に進むとき、メモリ上で何個分進むか** を表す数

たとえば、2 行 3 列のテンソル `[[0, 1, 2], [3, 4, 5]]` なら、

- 「列方向（右隣）」に 1 つ進むときは、メモリ上でも 1 個分進む → 列方向のストライドは 1
- 「行方向（下の行）」に 1 つ進むときは、メモリ上で 3 個分進む（1 行につき 3 要素あるから）→ 行方向のストライドは 3

つまり **stride は「各次元の隣に行くときのメモリの歩幅」** だと思うとイメージしやすいです。

この stride の並び方が「素直」なとき（たとえば行方向のストライドが列数、列方向のストライドが 1）が、PyTorch のいう「メモリ上で連続（contiguous）」な状態です。転置などをすると、この stride のパターンが変わり、non-contiguous になります。

In [14]:
import torch

# 2 行 3 列のテンソル
x = torch.tensor([
    [0, 1, 2],
    [3, 4, 5]
])

print("x =")
print(x)
print("x.shape:", x.shape)
print("x.stride():", x.stride())  # (行方向のストライド, 列方向のストライド)

# 転置してみる
y = x.t()
print("\ny = x.t() =")
print(y)
print("y.shape:", y.shape)
print("y.stride():", y.stride())

# それぞれの stride の意味:
# - x: (3, 1) → 行を 1 つ下に行くとメモリ上で 3 個進む／列を 1 つ右に行くと 1 個進む
# - y: (1, 3) → 行を 1 つ下に行くとメモリ上で 1 個進む／列を 1 つ右に行くと 3 個進む

print("\nx.is_contiguous():", x.is_contiguous())
print("y.is_contiguous():", y.is_contiguous())

x =
tensor([[0, 1, 2],
        [3, 4, 5]])
x.shape: torch.Size([2, 3])
x.stride(): (3, 1)

y = x.t() =
tensor([[0, 3],
        [1, 4],
        [2, 5]])
y.shape: torch.Size([3, 2])
y.stride(): (1, 3)

x.is_contiguous(): True
y.is_contiguous(): False


### 転置以外で「非連続」になるパターン

転置（`.t()`）のほかにも、次のような操作をするとテンソルが non-contiguous になることがあります。

- **スライスで「飛び飛び」に取る**  
  `x[:, ::2]` のようにステップを指定すると、メモリ上でとびとびに読む形になり、非連続になりやすいです。
- **1 列だけ・1 行だけ取り出す**  
  `x[:, 1]`（ある列だけ）や `x[0, :]`（ある行だけ）は、元の 2 次元の並びのまま「1 本の線」で見ているだけなので、ストライドが 1 でなくなり、非連続になります。
- **`permute()` で次元の順番を入れ替える**  
  3 次元以上のテンソルで `permute(2, 0, 1)` のように軸の順序を変えると、転置の拡張のような状態になり、多くの場合 non-contiguous になります。

いずれも「データのコピーはせず、見え方（と stride）だけ変えている」ため、`is_contiguous()` が `False` になることがあります。

In [15]:
import torch

# 3×4 のテンソルを用意
a = torch.arange(12).view(3, 4)
print("a =")
print(a)
print("a.stride():", a.stride(), "  a.is_contiguous():", a.is_contiguous())

# パターン1: 1 列だけ取り出す → 行方向にストライドが効くので非連続
col1 = a[:, 1]
print("\ncol1 = a[:, 1] =", col1)
print("col1.stride():", col1.stride(), "  col1.is_contiguous():", col1.is_contiguous())

# パターン2: ステップ付きスライス（1 列おきに取る）→ 飛び飛びなので非連続
every_other = a[:, ::2]
print("\nevery_other = a[:, ::2] =")
print(every_other)
print("every_other.stride():", every_other.stride(), "  every_other.is_contiguous():", every_other.is_contiguous())

# パターン3: 3 次元テンソルで permute して次元の順序を変える → 非連続になりやすい
b = torch.arange(24).view(2, 3, 4)
c = b.permute(2, 0, 1)  # 軸の順序を (2,0,1) に変更
print("\nb.shape:", b.shape, "  b.is_contiguous():", b.is_contiguous())
print("c = b.permute(2, 0, 1)  →  c.shape:", c.shape, "  c.is_contiguous():", c.is_contiguous())

a =
tensor([[ 0,  1,  2,  3],
        [ 4,  5,  6,  7],
        [ 8,  9, 10, 11]])
a.stride(): (4, 1)   a.is_contiguous(): True

col1 = a[:, 1] = tensor([1, 5, 9])
col1.stride(): (4,)   col1.is_contiguous(): False

every_other = a[:, ::2] =
tensor([[ 0,  2],
        [ 4,  6],
        [ 8, 10]])
every_other.stride(): (4, 2)   every_other.is_contiguous(): False

b.shape: torch.Size([2, 3, 4])   b.is_contiguous(): True
c = b.permute(2, 0, 1)  →  c.shape: torch.Size([4, 2, 3])   c.is_contiguous(): False


## 「メモリ上で連続（contiguous）」って何？

PyTorch では、テンソルが **メモリ上で連続して並んでいるかどうか** を意識しています。これを `is_contiguous()` で確認できます。

- **連続（contiguous）なテンソル**
  - 要素がメモリ上で「ずらーっと 1 本の配列」のように並んでいる状態
  - `view` など、一部のメソッドは **連続していることが前提** です。
- **非連続（non-contiguous）なテンソル**
  - たとえば「転置（transpose）」や「ストライド付きのスライス」などを行うと、
    実データはそのままで、**見え方だけを変える** 場合があります。
  - このとき、内部的には「飛び飛びにメモリを読む」ような形になるため、
    テンソルは "non-contiguous" になります。

イメージとしては、

- 連続: `[0, 1, 2, 3, 4, 5]` がそのまま 0 番地から 5 番地まで並んでいる
- 非連続: 元の配列はそのままだけど、「2 個おきに拾った見た目」だけを別のテンソルとして見ている

というような違いだと思うと分かりやすいです。

**補足**: `clone()` はデータをコピーしますが、メモリの並び（ストライド）はそのままなので、非連続テンソルを `clone()` しても結果は非連続のままです。連続にしたいときは `.contiguous()` を使います。

このあと `view` を説明するときに、
「**なぜ連続でないと `view` がエラーになるのか**」という話にもつながります。

In [16]:
import torch

# まずはシンプルな 2 次元テンソル
x = torch.tensor([
    [0, 1, 2],
    [3, 4, 5]
])
print("x =")
print(x)
print("x.is_contiguous():", x.is_contiguous())

# 転置（transpose）してみると…
y = x.t()
print("\ny = x.t() =")
print(y)
print("y.is_contiguous():", y.is_contiguous())

# clone() はデータをコピーするが、メモリの並び（レイアウト）はそのままなので、
# 非連続テンソルを clone しても z はやはり non-contiguous のまま
z = y.clone()
print("\nz = y.clone() =")
print(z)
print("z.is_contiguous():", z.is_contiguous())

# 連続にしたいときは .contiguous() を使う（必要ならメモリ上で並び直す）
w = y.contiguous()
print("\nw = y.contiguous() =")
print(w)
print("w.is_contiguous():", w.is_contiguous())

x =
tensor([[0, 1, 2],
        [3, 4, 5]])
x.is_contiguous(): True

y = x.t() =
tensor([[0, 3],
        [1, 4],
        [2, 5]])
y.is_contiguous(): False

z = y.clone() =
tensor([[0, 3],
        [1, 4],
        [2, 5]])
z.is_contiguous(): False

w = y.contiguous() =
tensor([[0, 3],
        [1, 4],
        [2, 5]])
w.is_contiguous(): True


## `view` メソッド（テンソルの形を変える）

`view` は、**テンソルの「形（shape）」だけを変えるメソッド**です。

- 書式: `x.view(new_shape)` または `x.view(dim1, dim2, ...)`
- 中身の要素（値）はそのままで、**並び順も変えずに** 見かけの形だけ変えます。
- `-1` を使うと、「ここは自動で計算して埋めてね」という意味になります。

### よくある使い方のイメージ

- バッチサイズ $batch\_size$ のベクトルを「列ベクトル」にしたいとき:  
  `actions.view(-1, 1)`  
  → これは「$batch\_size \times 1$ の 2 次元テンソル」に変形する、という意味になります。

In [17]:
import torch

# 1 次元テンソル（要素数 6）
x = torch.tensor([0, 1, 2, 3, 4, 5])
print("x:", x)
print("x.shape:", x.shape)

# 2 行 3 列に変形
x_2x3 = x.view(2, 3)
print("\nx_2x3 = x.view(2, 3):")
print(x_2x3)
print("x_2x3.shape:", x_2x3.shape)

# 3 行 2 列に変形
x_3x2 = x.view(3, 2)
print("\nx_3x2 = x.view(3, 2):")
print(x_3x2)
print("x_3x2.shape:", x_3x2.shape)

# -1 を使って、自動計算してもらう例
x_auto = x.view(-1, 3)  # 「行数はおまかせ、列は 3」
print("\nx_auto = x.view(-1, 3):")
print(x_auto)
print("x_auto.shape:", x_auto.shape)

x: tensor([0, 1, 2, 3, 4, 5])
x.shape: torch.Size([6])

x_2x3 = x.view(2, 3):
tensor([[0, 1, 2],
        [3, 4, 5]])
x_2x3.shape: torch.Size([2, 3])

x_3x2 = x.view(3, 2):
tensor([[0, 1],
        [2, 3],
        [4, 5]])
x_3x2.shape: torch.Size([3, 2])

x_auto = x.view(-1, 3):
tensor([[0, 1, 2],
        [3, 4, 5]])
x_auto.shape: torch.Size([2, 3])


## `view` と `reshape` の違い

`view` と `reshape` は、どちらも「**テンソルの形を変える**」ためのメソッドです。多くの場合は、**結果は同じ** になりますが、内部的な動作に違いがあります。

- **`view`**
  - テンソルがメモリ上で **連続（contiguous）** に並んでいる場合にだけ使えます。
  - 連続でない場合に `view` しようとすると、`RuntimeError: view size is not compatible with ...` のようなエラーになります。

- **`reshape`**
  - できるだけ `view` と同じように動こうとしますが、
  - 必要であれば **内部でコピーを作って** 形を変えることがあります。
  - そのため、「`view` ではエラーになる場合でも、`reshape` は通る」ことがあります。

実務的には、

- **「連続かどうかを意識していて、できるだけコピーを避けたい」→ `view` を使う**
- **「そこまで気にせず、とにかく形だけ変えたい」→ `reshape` を使う**

という感覚で覚えておくと分かりやすいです。

In [18]:
import torch

x = torch.arange(12)  # [0, 1, ..., 11]
print("x:", x)
print("x.shape:", x.shape)

# view と reshape は、この程度なら同じ結果
x_view = x.view(3, 4)
x_reshape = x.reshape(3, 4)
print("\nx_view = x.view(3, 4):")
print(x_view)
print("x_reshape = x.reshape(3, 4):")
print(x_reshape)

# メモリが連続でないテンソルを作る（転置など）
y = x_view.t()  # 転置（transpose）すると、非連続になりやすい
print("\ny = x_view.t():")
print(y)
print("y.is_contiguous():", y.is_contiguous())

# 連続でないテンソルに対して view すると、エラーになることがある
try:
    y_view = y.view(2, 6)
    print("\ny_view = y.view(2, 6):")
    print(y_view)
except RuntimeError as e:
    print("\nview でエラーが出る例:")
    print(e)

# reshape は必要に応じてコピーを作って対応してくれる
y_reshape = y.reshape(2, 6)
print("\ny_reshape = y.reshape(2, 6):")
print(y_reshape)

x: tensor([ 0,  1,  2,  3,  4,  5,  6,  7,  8,  9, 10, 11])
x.shape: torch.Size([12])

x_view = x.view(3, 4):
tensor([[ 0,  1,  2,  3],
        [ 4,  5,  6,  7],
        [ 8,  9, 10, 11]])
x_reshape = x.reshape(3, 4):
tensor([[ 0,  1,  2,  3],
        [ 4,  5,  6,  7],
        [ 8,  9, 10, 11]])

y = x_view.t():
tensor([[ 0,  4,  8],
        [ 1,  5,  9],
        [ 2,  6, 10],
        [ 3,  7, 11]])
y.is_contiguous(): False

view でエラーが出る例:
view size is not compatible with input tensor's size and stride (at least one dimension spans across two contiguous subspaces). Use .reshape(...) instead.

y_reshape = y.reshape(2, 6):
tensor([[ 0,  4,  8,  1,  5,  9],
        [ 2,  6, 10,  3,  7, 11]])


## PyTorch の `gather` メソッド

`gather` は、「**インデックスのテンソルを使って、元のテンソルから値を抜き出す**」ためのメソッドです。

PyTorch では、次の **2 通りの呼び出し方** ができます。

1. **関数として呼び出すパターン**  
   - 書式: `torch.gather(input, dim, index)`
   - 例: `out = torch.gather(x, dim=1, index=index)`

2. **テンソルのメソッドとして呼び出すパターン**  
   - 書式: `input.gather(dim, index)`
   - 例: `out = x.gather(dim=1, index=index)`

どちらも **結果は同じ** で、`input` と `dim`、`index` をどう指定するかだけ理解しておけば OK です。

- **引数のおさらい**
  - **`input`**: 元のテンソル（メソッド呼び出しの場合は `self` に相当）
  - **`dim`**: どの次元に沿って要素を取り出すか（0 や 1 などの整数）
  - **`index`**: 取り出したい位置を示すテンソル
- **重要なポイント**
  - `index` の形状は、**出力の形状と同じ**になります。
  - `index` は、`input` と同じ形状か、少なくとも `dim` 以外の次元については同じ形状である必要があります。

強化学習では、Q 値テーブル `q_values` から「実際に選んだ行動の Q 値だけを抜き出す」ときなどによく使われます。

In [19]:
import torch

# 入力テンソル（例）: 2 行 3 列
x = torch.tensor([
    [10, 20, 30],
    [40, 50, 60]
])

# dim=1（列方向）に沿って値を取り出したいとする
# 各行ごとに「どの列を取り出すか」を index で指定する
index = torch.tensor([
    [2, 1],  # 1 行目からは列 2 と列 1 を取り出す
    [0, 0]   # 2 行目からは列 0 と列 0 を取り出す
])

# gather によって、index で指定した位置の値を集める
# 1) 関数として呼び出すパターン
out_func = torch.gather(x, dim=1, index=index)

# 2) テンソルのメソッドとして呼び出すパターン
out_method = x.gather(dim=1, index=index)

print("x =")
print(x)
print("\nindex =")
print(index)
print("\nout_func = torch.gather(x, dim=1, index=index) =")
print(out_func)
print("\nout_method = x.gather(dim=1, index=index) =")
print(out_method)

x =
tensor([[10, 20, 30],
        [40, 50, 60]])

index =
tensor([[2, 1],
        [0, 0]])

out_func = torch.gather(x, dim=1, index=index) =
tensor([[30, 20],
        [40, 40]])

out_method = x.gather(dim=1, index=index) =
tensor([[30, 20],
        [40, 40]])


In [20]:
import torch

# 強化学習でのイメージ例
# q_values: 各状態ごとの行動価値（バッチサイズ=3, 行動数=4）
q_values = torch.tensor([
    [1.0, 2.0, 3.0, 4.0],
    [0.5, 1.5, 2.5, 3.5],
    [10.0, 20.0, 30.0, 40.0]
])

# 各状態で実際に選んだ行動（例）
actions = torch.tensor([2, 0, 3])  # 0〜3 の整数

# gather を使うために、actions を 2 次元に変形（列ベクトル）
actions_2d = actions.view(-1, 1)  # shape: (batch, 1)

# dim=1（行動の次元）で、選んだ行動の Q 値だけを抜き出す
chosen_q_values = torch.gather(q_values, dim=1, index=actions_2d)

print("q_values =")
print(q_values)
print("\nactions =")
print(actions)
print("\nactions_2d =")
print(actions_2d)
print("\nchosen_q_values =")
print(chosen_q_values)  # shape: (batch, 1)

q_values =
tensor([[ 1.0000,  2.0000,  3.0000,  4.0000],
        [ 0.5000,  1.5000,  2.5000,  3.5000],
        [10.0000, 20.0000, 30.0000, 40.0000]])

actions =
tensor([2, 0, 3])

actions_2d =
tensor([[2],
        [0],
        [3]])

chosen_q_values =
tensor([[ 3.0000],
        [ 0.5000],
        [40.0000]])
